# Day 5 — Lists, Dicts, Sets in Depth

> ⚠️ **Why this matters.** Today your english-helper grows up. It stops being a script that runs once and forgets — it **remembers everything between runs.** Words saved today are still there tomorrow. That's the difference between an exercise and a tool.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jakkzz/prince-curriculum/blob/main/phase-1-python-cli/lessons/05-lists-dicts-sets.ipynb)

## What you'll do today

**Time:** 90 min lesson + 60 min mini-project + 45 min quiz.

By the end:

- [ ] You know the killer methods of `list`, `dict`, `set`
- [ ] You can choose the right collection for each job
- [ ] You can read and write JSON files
- [ ] You can build a tiny interactive REPL-style CLI
- [ ] You've shipped `vocab_app.py` — your **first persistent** english-helper

## The mental model

Collections you've met:

| Type | Lookup speed | Order | Mutable | Duplicates | Use for |
|------|--------------|-------|---------|------------|---------|
| `list` | O(n) | yes | yes | yes | Ordered series; you'll iterate often |
| `tuple` | O(n) | yes | no | yes | Fixed-size related group |
| `set` | O(1) | no | yes | no | Membership tests; unique values |
| `frozenset` | O(1) | no | no | no | Hashable set (rare) |
| `dict` | O(1) | yes (insertion) | yes | n/a | Key-value lookup |

**Today's headline:** dicts and sets are O(1) for membership checks. Lists are O(n). The data structure you pick determines how fast your program runs.

## 1. List methods you'll actually use

In [1]:
words = ['thorough', 'ubiquitous']

words.append('nuance')         # add to end
words.insert(0, 'a')           # add at position
words.extend(['x', 'y'])       # add multiple
print(words)

words.remove('a')              # remove first match
popped = words.pop()           # remove and return last
print('popped:', popped)
print(words)

print('index of nuance:', words.index('nuance'))
print('count of x:', words.count('x'))

['a', 'thorough', 'ubiquitous', 'nuance', 'x', 'y']
popped: y
['thorough', 'ubiquitous', 'nuance', 'x']
index of nuance: 2
count of x: 0


In [1]:
# Sorting
nums = [3, 1, 4, 1, 5, 9, 2, 6]
sorted(nums)              # returns a new sorted list


[1, 1, 2, 3, 4, 5, 6, 9]

In [1]:
nums = [3, 1, 4, 1, 5]
nums.sort()               # in-place — modifies the original
nums

[1, 1, 3, 4, 5]

> ⚠️ **`sort()` vs `sorted()`.** `sort()` modifies the list in place and returns `None`. `sorted()` returns a new list. **Mixing them up is a classic bug** — `nums = nums.sort()` assigns `None` to `nums`. Watch for it.

In [1]:
# Sort with a key
words = ['ubiquitous', 'nuance', 'thorough']
sorted(words, key=len)    # by length

['nuance', 'thorough', 'ubiquitous']

## 2. Dict methods you'll actually use

In [1]:
user = {'name': 'Prince', 'age': 16}

# Access
print(user['name'])                     # KeyError if missing
print(user.get('email'))                # None if missing
print(user.get('email', 'no email'))    # custom default

# Iterate
for key in user:
    print(f'  {key}')
for key, value in user.items():
    print(f'  {key}={value}')

# Add/update
user['email'] = 'prince@example.com'
user.update({'age': 17, 'country': 'TH'})
print(user)

# Delete
del user['country']
removed = user.pop('age')
print(user)
print('removed:', removed)

Prince
None
no email
  name
  age
  name=Prince
  age=16
{'name': 'Prince', 'age': 17, 'email': 'prince@example.com', 'country': 'TH'}
{'name': 'Prince', 'email': 'prince@example.com'}
removed: 17


## 3. Set methods + set algebra

In [1]:
a = {'apple', 'banana', 'cherry'}
b = {'banana', 'cherry', 'date'}

# Algebra
print('union:', a | b)              # all in A or B
print('intersection:', a & b)        # all in BOTH
print('A - B:', a - b)               # in A but not B
print('symmetric diff:', a ^ b)      # in one but not both

# Membership — O(1), fast even for huge sets
print('apple in a:', 'apple' in a)
print('mango in a:', 'mango' in a)

# Add/remove
a.add('elderberry')
a.discard('apple')          # no error if missing
# a.remove('apple')         # KeyError if missing
print(a)

union: {'apple', 'banana', 'cherry', 'date'}
intersection: {'banana', 'cherry'}
A - B: {'apple'}
symmetric diff: {'apple', 'date'}
apple in a: True
mango in a: False
{'cherry', 'banana', 'elderberry'}


### Set use case: deduplicate a list

Idiom: `list(set(xs))` — strips duplicates. Loses order; if order matters, see below.

In [1]:
tags = ['python', 'web', 'python', 'sql', 'web']
list(set(tags))

['web', 'python', 'sql']

Order-preserving dedup (Python 3.7+ since dicts are ordered):

In [1]:
tags = ['python', 'web', 'python', 'sql', 'web']
list(dict.fromkeys(tags))

['python', 'web', 'sql']

## 4. Comprehensions in all shapes

List, dict, set comprehensions — same pattern: `[expr for item in iterable if cond]`.

In [1]:
# Dict comprehension
words = ['thorough', 'nuance', 'ubiquitous']
{w: len(w) for w in words}

{'thorough': 8, 'nuance': 6, 'ubiquitous': 10}

In [1]:
# Set comprehension — unique first letters
{w[0] for w in words}

{'t', 'n', 'u'}

In [1]:
# Inverting a dict
ipa_lookup = {'thorough': '/ˈθʌrə/', 'nuance': '/ˈnuːɑːns/'}
{ipa: word for word, ipa in ipa_lookup.items()}

{'/ˈθʌrə/': 'thorough', '/ˈnuːɑːns/': 'nuance'}

## 5. JSON — persistent storage in 3 lines

Python and JSON are very similar. `dict ↔ object`, `list ↔ array`, `None ↔ null`, etc. The `json` module converts between them.

```python
import json

# Python → JSON string
json.dumps({'a': 1, 'b': [1, 2]})
# → '{"a": 1, "b": [1, 2]}'

# JSON string → Python
json.loads('{"a": 1}')
# → {'a': 1}
```

For files, use `pathlib`:

In [ ]:
from pathlib import Path
import json

data = {
    'words': [
        {'word': 'thorough', 'ipa': '/ˈθʌrə/', 'thai': 'ละเอียด'},
        {'word': 'nuance', 'ipa': '/ˈnuːɑːns/', 'thai': 'ความละเอียดอ่อน'},
    ]
}

# Write
Path('/tmp/test.json').write_text(json.dumps(data, ensure_ascii=False, indent=2))

# Read
loaded = json.loads(Path('/tmp/test.json').read_text())
print(loaded)

**Important flags:**

- `ensure_ascii=False` — keeps Thai/Unicode chars as-is (otherwise they get escaped to `\u...`).
- `indent=2` — pretty-print so humans can read the file.


## End-of-day mini-project — `vocab_app.py`

> 🎯 **Today's piece of [English Helper](RUNNING-PROJECT.md):** the **first persistent** version. Add words, look them up, quiz yourself — and everything is saved to disk between runs. From here on, your tool **remembers**.

### What you're building

An interactive CLI that reads/writes a JSON file at `~/.english-helper/words.json`:

```
$ uv run python vocab_app.py

🎯 English Helper — type 'help' for commands

> add thorough /ˈθʌrə/ ละเอียด
Added 'thorough'.

> add nuance /ˈnuːɑːns/ ความละเอียดอ่อน
Added 'nuance'.

> list
thorough     /ˈθʌrə/        ละเอียด
nuance       /ˈnuːɑːns/     ความละเอียดอ่อน
Total: 2

> stats
Words in dictionary: 2
Unique starting letters: 2

> quit
Saved.

$ uv run python vocab_app.py
🎯 English Helper — type 'help' for commands
> list
thorough     /ˈθʌrə/        ละเอียด    ← still here!
nuance       /ˈnuːɑːns/     ความละเอียดอ่อน
Total: 2
```

### Required commands

| Command | Behavior |
|---------|----------|
| `add WORD IPA THAI` | Add a word. Reject duplicates. |
| `remove WORD` | Remove. Say if it wasn't there. |
| `lookup WORD` | Show details or 'not found'. |
| `list` | Print all words. |
| `stats` | Show counts: total words, unique first letters, longest word. |
| `help` | List commands. |
| `quit` | Save and exit. |

### Requirements

- Storage: `~/.english-helper/words.json`. Create directory if missing.
- Use a **dict** keyed by `word` (not a list) for O(1) lookup. Each value is `{'ipa': ..., 'thai': ...}`.
- All functions type-hinted. Pass `mypy`.
- Import `lookup` etc. from your Day 4 `dictionary.py` if it makes sense — or write fresh ones (dict-keyed is more elegant for this use).
- Use a `while True:` loop with `input()` for the REPL.
- Save **on every command** (paranoid) OR on `quit` only (faster but you could lose changes if you Ctrl-C). Your call — document it in a comment.

### Try it

In [ ]:
# Sketch the storage helpers and command dispatch here.

from pathlib import Path
import json

DATA_FILE = Path.home() / ".english-helper" / "words.json"

# Your code


<details>
<summary>Solution — try first!</summary>

```python
# vocab_app.py
"""english-helper — persistent vocabulary app.

Stores words in ~/.english-helper/words.json. Saves on every command.
"""
import json
from pathlib import Path

DATA_FILE = Path.home() / ".english-helper" / "words.json"

WordEntry = dict[str, str]
WordStore = dict[str, WordEntry]


def load() -> WordStore:
    if not DATA_FILE.exists():
        return {}
    return json.loads(DATA_FILE.read_text())


def save(store: WordStore) -> None:
    DATA_FILE.parent.mkdir(exist_ok=True)
    DATA_FILE.write_text(json.dumps(store, ensure_ascii=False, indent=2))


def cmd_add(store: WordStore, args: list[str]) -> str:
    if len(args) < 3:
        return 'Usage: add WORD IPA THAI'
    word, ipa, thai = args[0], args[1], ' '.join(args[2:])
    if word in store:
        return f'{word!r} already exists. Use remove first.'
    store[word] = {'ipa': ipa, 'thai': thai}
    return f'Added {word!r}.'


def cmd_remove(store: WordStore, args: list[str]) -> str:
    if not args:
        return 'Usage: remove WORD'
    word = args[0]
    if word not in store:
        return f'{word!r} not found.'
    del store[word]
    return f'Removed {word!r}.'


def cmd_lookup(store: WordStore, args: list[str]) -> str:
    if not args:
        return 'Usage: lookup WORD'
    word = args[0]
    entry = store.get(word)
    if not entry:
        return f'{word!r} not found.'
    return f"{word}  {entry['ipa']}  {entry['thai']}"


def cmd_list(store: WordStore, args: list[str]) -> str:
    if not store:
        return '(empty — try `add`)'
    lines = [
        f"{w:15} {e['ipa']:25} {e['thai']}"
        for w, e in sorted(store.items())
    ]
    lines.append(f'Total: {len(store)}')
    return '\n'.join(lines)


def cmd_stats(store: WordStore, args: list[str]) -> str:
    if not store:
        return 'No words yet.'
    first_letters = {w[0] for w in store}
    longest = max(store, key=len)
    return (
        f'Words in dictionary: {len(store)}\n'
        f'Unique starting letters: {len(first_letters)}\n'
        f'Longest word: {longest!r} ({len(longest)} letters)'
    )


def cmd_help(store: WordStore, args: list[str]) -> str:
    return (
        'Commands:\n'
        '  add WORD IPA THAI    Add a word\n'
        '  remove WORD          Remove a word\n'
        '  lookup WORD          Show one\n'
        '  list                 Show all\n'
        '  stats                Counts\n'
        '  quit                 Save and exit'
    )


COMMANDS = {
    'add': cmd_add, 'remove': cmd_remove, 'lookup': cmd_lookup,
    'list': cmd_list, 'stats': cmd_stats, 'help': cmd_help,
}


def main() -> None:
    store = load()
    print("🎯 English Helper — type 'help' for commands")
    while True:
        try:
            line = input('> ').strip()
        except (EOFError, KeyboardInterrupt):
            print()
            break
        if not line:
            continue
        parts = line.split()
        cmd, args = parts[0], parts[1:]
        if cmd == 'quit':
            break
        handler = COMMANDS.get(cmd)
        if not handler:
            print(f"Unknown: {cmd!r}. Try 'help'.")
            continue
        print(handler(store, args))
        save(store)   # paranoid: save after every successful command
    save(store)
    print('Saved.')


if __name__ == '__main__':
    main()
```

**Stretch:**
- Add a `search PATTERN` command using `in` (`'th' in word`).
- Add a `quiz` command that uses your Day 4 `dictionary.random_entry`.
- Add an `export FILENAME` command — dumps to a CSV.
- Add timestamps: each entry tracks when added.
</details>

## Connect to the project

> 🎯 **Connects to the project:** This is the end of Phase 1 Week 1. You now have:
> - `vocab_card.py` — print a single card
> - `word_list.py` — typed list of words
> - `pronunciation_quiz.py` — interactive quiz
> - `dictionary.py` — clean function module
> - `vocab_app.py` — persistent CLI

> Together they're a **working English-learning tool**. Add it to your daily routine: 5 new words a week, quiz yourself before bed. Use the thing you built.

> Week 2 levels everything up: we add the `requests` library and start pulling **real definitions from the [Free Dictionary API](https://dictionaryapi.dev/)**. Type a new word, get its real IPA, sample sentences, and audio pronunciation URL — auto-filled.

## Self-check

<details>
<summary>1. When is a set better than a list?</summary>

When you need fast `in` checks (set is O(1), list is O(n)) AND/OR you want auto-deduplication. Loses ordering — if order matters, use a list (or `dict.fromkeys`).
</details>

<details>
<summary>2. <code>sort()</code> vs <code>sorted()</code> — what's the difference?</summary>

`sort()` modifies the list in place, returns `None`. `sorted()` returns a new sorted list, original unchanged. `xs = xs.sort()` is the bug — `xs` becomes None.
</details>

<details>
<summary>3. How do you safely look up a key in a dict?</summary>

`d.get(key)` returns None if missing, or `d.get(key, default)` for a custom default. Use `d[key]` only when you're sure the key exists.
</details>

<details>
<summary>4. What does <code>list(dict.fromkeys(xs))</code> do?</summary>

Removes duplicates from `xs` while preserving order (since dicts preserve insertion order in modern Python). `list(set(xs))` also dedupes but loses order.
</details>

<details>
<summary>5. Why <code>ensure_ascii=False</code> in <code>json.dumps</code>?</summary>

By default, json.dumps escapes non-ASCII characters as `\u…` sequences. Setting `ensure_ascii=False` keeps Thai characters (and other Unicode) as readable text in the JSON file.
</details>

## What's next

Tomorrow you start **Week 2**: testing with pytest, then we add `requests` and start hitting real APIs. Your english-helper goes online.

**Quiz:** [05-lists-dicts-sets-quiz.ipynb](05-lists-dicts-sets-quiz.ipynb)

**Week 1 commit checklist:**
- [ ] All 5 mini-projects committed and pushed
- [ ] All 5 quizzes taken; scores logged
- [ ] `vocab_app.py` actually works — you've added 5+ real words you want to learn
- [ ] Friday mentor check-in scheduled